## CHyMErA screen

https://www.nature.com/articles/s41587-020-0437-z

- 1344 Human paralogs
- 22 manually selected gene pairs of interest
- HAP1 and RPE1 cells

**Input:** Supplemental Table 8 - contains several sheets of interest: Summary, HAP1.T12, HAP1.T18, RPE1.T18, RPE1.T24

**Output:** Paralog pairs from screen annotated with negative GI hits and observed LFC

In [74]:
# import modules
import os
import re
import pandas as pd
import numpy as np
from natsort import natsorted

In [75]:
# set the base directory for the project
cwd = os.getcwd()
BASE_DIR = os.path.abspath(os.path.join(cwd, "..", ".."))

# build paths inside the repo
get_data_path = lambda folders, fname: os.path.normpath(
    os.path.join(BASE_DIR, *folders, fname)
)

file_path_tables8 = get_data_path(['data', 'input', 'CRISPR_screens'], 'chymera_table8.xlsx')

file_path_genenames = get_data_path(['data', 'input', 'other'], 'approved_and_previous_symbols.csv')

file_path_sample_info = get_data_path(['data', 'input', 'DepMap22Q4'], 'sample_info.csv')

file_path_processed_chymera_df = get_data_path(['data', 'output', 'processed_CRISPR_screens'], 'processed_chymera_df.csv')

In [76]:
# read input files
s8 = pd.read_excel(file_path_tables8, sheet_name="Summary", skiprows=1)
s8 = s8.drop(columns=['torin_gi_type', 'torin_early_effect_size', 'torin_late_effect_size', 'HAP1 +/- Torin'])
s8[:3]

,gene1,gene2,hap1_gi_type,rpe1_gi_type,hap1_early_effect_size,hap1_late_effect_size,rpe1_early_effect_size,rpe1_late_effect_size
0,SEC23A,SEC23B,shared,shared,-3.028953,-3.424526,-2.306238,-2.103825
1,SAR1A,SAR1B,shared,unique_early,-2.890827,-3.288142,-1.680115,-1.392617
2,COQ10B,COQ10A,shared,none,-2.869043,-3.114457,-0.990499,-0.992759


In [77]:
# Call negative GI if the effect is seen at either time point (shared), and effect size is negative at both time points

screen_hits = s8.assign(
    rpe1_hit = s8.apply(lambda x: ((x.rpe1_gi_type!='none') 
                        and (x.rpe1_late_effect_size < 0) 
                        and (x.rpe1_early_effect_size < 0)), axis=1),
    hap1_hit = s8.apply(lambda x: ((x.hap1_gi_type!='none') 
                        and (x.hap1_late_effect_size < 0) 
                        and (x.hap1_early_effect_size < 0)), axis=1)
)

In [78]:
print('Num RPE1 SL:', sum(screen_hits.rpe1_hit), '/', screen_hits.shape[0])
print('Num HAP1 SL:', sum(screen_hits.hap1_hit), '/', screen_hits.shape[0])

Num RPE1 SL: 107 / 688
Num HAP1 SL: 214 / 688


In [79]:
screen_hits = screen_hits[['gene1','gene2','rpe1_hit', 'hap1_hit']]
screen_hits = screen_hits.rename(columns={'gene1':'A1','gene2':'A2'}) #  , 'rpe1_hit':'SL'})
screen_hits[:3]

,A1,A2,rpe1_hit,hap1_hit
0,SEC23A,SEC23B,True,True
1,SAR1A,SAR1B,True,True
2,COQ10B,COQ10A,False,True


## Add Entrez ID and DepMap ID to the dataset

In [80]:
# read the gene names mapping file
id_map = pd.read_csv(file_path_genenames)

# create dictionaries to map gene symbols to Entrez IDs
approved_sym_to_entrez_id = dict(zip(id_map['Approved symbol'], id_map['entrez_id']))

# create dictionaries to map previous gene symbols to Entrez IDs
id_map_cleaned = id_map.dropna(axis=0, how='any', subset=['Previous symbol', 'entrez_id']).reset_index(drop=True)
prev_sym_to_entrez_id = dict(zip(id_map_cleaned['Previous symbol'], id_map_cleaned['entrez_id']))

In [81]:
# create a df from the target genes 
gene_symbols_s8 = pd.concat([screen_hits['A1'], screen_hits['A2']]).unique().tolist()
d = {'symbols' : gene_symbols_s8}
gene_symbols_s8_df = pd.DataFrame(data=d)

gene_symbols_s8_df['entrez_id'] = gene_symbols_s8_df['symbols'].map(approved_sym_to_entrez_id)
gene_symbols_s8_df['entrez_id'] = gene_symbols_s8_df['entrez_id'].fillna(gene_symbols_s8_df['symbols'].map(prev_sym_to_entrez_id))

gene_symbols_s8_df = gene_symbols_s8_df.dropna(subset=['entrez_id']).reset_index(drop=True)
gene_symbols_s8_df['entrez_id'] = gene_symbols_s8_df['entrez_id'].astype(int)

gene_symbols_s8_df_dict = dict(zip(gene_symbols_s8_df['symbols'], gene_symbols_s8_df['entrez_id']))

In [82]:
screen_hits.rename(columns={'A1': 'org_A1', 'A2':'org_A2'}, inplace=True)

In [83]:
screen_hits.insert(0, 'A1_entrez', screen_hits['org_A1'].map(gene_symbols_s8_df_dict))
screen_hits.insert(1, 'A2_entrez', screen_hits['org_A2'].map(gene_symbols_s8_df_dict))

In [84]:
entrez_id_to_approved_sym = dict(zip(id_map['entrez_id'], id_map['Approved symbol']))
screen_hits.insert(2, 'A1', screen_hits['A1_entrez'].map(entrez_id_to_approved_sym))
screen_hits.insert(3, 'A2', screen_hits['A2_entrez'].map(entrez_id_to_approved_sym))

In [85]:
screen_hits[:3]

,A1_entrez,A2_entrez,A1,A2,org_A1,org_A2,rpe1_hit,hap1_hit
0,10484,10483,SEC23A,SEC23B,SEC23A,SEC23B,True,True
1,56681,51128,SAR1A,SAR1B,SAR1A,SAR1B,True,True
2,80219,93058,COQ10B,COQ10A,COQ10B,COQ10A,False,True


In [86]:
#screen_hits_melt = pd.melt(screen_hits, id_vars=['A1', 'A2', 'A1_entrez', 'A2_entrez', 'org_A1', 'org_A2'], 
#                           value_vars=screen_hits.columns[6:], 
#                           var_name='cell_line', value_name='Hit')

In [87]:
# Sorted gene pair column
list_c = [[x, y] for x, y in zip(screen_hits.A1, screen_hits.A2)]

genepairs = []
for pair in list_c:
    sorted_pair = natsorted(pair)
    genepairs.append(sorted_pair)

m = []
for i in range(0 , len(genepairs)):
    a = '_'.join(genepairs[i])
    m.append(a)

screen_hits.insert(0, "genepair", m)

In [88]:
# Sorted gene pair column
list_c = [[x, y] for x, y in zip(screen_hits.org_A1, screen_hits.org_A2)]

genepairs = []
for pair in list_c:
    sorted_pair = natsorted(pair)
    genepairs.append(sorted_pair)

m = []
for i in range(0 , len(genepairs)):
    a = '_'.join(genepairs[i])
    m.append(a)

screen_hits.insert(0, "org_genepair", m)

In [89]:
screen_hits_df = screen_hits.sort_values(by=['genepair'])
screen_hits_df = screen_hits_df.reset_index(drop=True)

In [90]:
screen_hits_df.loc[screen_hits_df['org_A1'] == 'FAM102A']

,org_genepair,genepair,A1_entrez,A2_entrez,A1,A2,org_A1,org_A2,rpe1_hit,hap1_hit
175,FAM102A_FAM102B,EEIG1_EEIG2,399665,284611,EEIG1,EEIG2,FAM102A,FAM102B,True,True


## Load observed LFC values

In [91]:
def load_observed_lfc(sheet_name,col_name,new_col_name):

    obs_lfc = pd.read_excel(file_path_tables8, sheet_name=sheet_name).drop_duplicates() # there are duplicate rows
    obs_lfc = obs_lfc.rename(columns={col_name:new_col_name}).reset_index(drop=True)

    pairlist = [[x, y] for x, y in zip(obs_lfc.gene1, obs_lfc.gene2)]

    genepairs = []
    for pair in pairlist:
        sorted_pair = natsorted(pair)
        genepairs.append(sorted_pair)
    
    m = []
    for i in range(0 , len(genepairs)):
        a = '_'.join(genepairs[i])
        m.append(a)
    
    obs_lfc.insert(0, "org_genepair", m)
    obs_lfc = obs_lfc[['org_genepair', new_col_name]]
    return obs_lfc

In [92]:
hap1_t1 = load_observed_lfc('HAP1.T12','early_mean_observed_LFC','hap1_early_obs_LFC')
hap1_t1[:1]

,org_genepair,hap1_early_obs_LFC
0,ABHD12_ABHD12B,-0.067548


In [93]:
hap1_t2 = load_observed_lfc('HAP1.T18','late_mean_observed_LFC','hap1_late_obs_LFC')
hap1_t2[:1]

,org_genepair,hap1_late_obs_LFC
0,ABHD12_ABHD12B,-0.130963


In [94]:
rpe1_t1 = load_observed_lfc('RPE1.T18','early_mean_observed_LFC','rpe1_early_obs_LFC')
rpe1_t1[:1]

,org_genepair,rpe1_early_obs_LFC
0,ABHD12_ABHD12B,-0.288012


In [95]:
rpe1_t2 = load_observed_lfc('RPE1.T24','late_mean_observed_LFC','rpe1_late_obs_LFC')
rpe1_t2[:1]

,org_genepair,rpe1_late_obs_LFC
0,ABHD12_ABHD12B,-0.300921


In [96]:
screen_pairs_w_lfc = pd.merge(screen_hits_df, hap1_t1)
screen_pairs_w_lfc = pd.merge(screen_pairs_w_lfc, hap1_t2)
screen_pairs_w_lfc = pd.merge(screen_pairs_w_lfc, rpe1_t1)
screen_pairs_w_lfc = pd.merge(screen_pairs_w_lfc, rpe1_t2)

display(screen_pairs_w_lfc[:4])
display(screen_pairs_w_lfc.loc[screen_pairs_w_lfc['org_A1'] == 'FAM102A'],)

,org_genepair,genepair,A1_entrez,A2_entrez,A1,A2,org_A1,org_A2,rpe1_hit,hap1_hit,hap1_early_obs_LFC,hap1_late_obs_LFC,rpe1_early_obs_LFC,rpe1_late_obs_LFC
0,ABHD12_ABHD12B,ABHD12_ABHD12B,26090,145447,ABHD12,ABHD12B,ABHD12,ABHD12B,False,False,-0.067548,-0.130963,-0.288012,-0.300921
1,ABL1_ABL2,ABL1_ABL2,25,27,ABL1,ABL2,ABL1,ABL2,True,False,0.068177,0.047309,0.059887,0.149824
2,ACBD3_TMED8,ACBD3_TMED8,64746,283578,ACBD3,TMED8,ACBD3,TMED8,False,False,0.228133,0.258238,-0.099709,-0.178376
3,ACBD7_DBI,ACBD7_DBI,1622,414149,DBI,ACBD7,DBI,ACBD7,True,True,-0.016598,0.152179,-0.384733,-0.521418


,org_genepair,genepair,A1_entrez,A2_entrez,A1,A2,org_A1,org_A2,rpe1_hit,hap1_hit,hap1_early_obs_LFC,hap1_late_obs_LFC,rpe1_early_obs_LFC,rpe1_late_obs_LFC
175,FAM102A_FAM102B,EEIG1_EEIG2,399665,284611,EEIG1,EEIG2,FAM102A,FAM102B,True,True,-0.266611,-0.480234,-0.908453,-1.313825


In [97]:
screen_pairs_w_lfc = screen_pairs_w_lfc.assign(
    min_hap1_lfc = screen_pairs_w_lfc.apply(lambda x: min(x.hap1_early_obs_LFC, x.hap1_late_obs_LFC), axis=1),
    min_rpe1_lfc = screen_pairs_w_lfc.apply(lambda x: min(x.rpe1_early_obs_LFC, x.rpe1_late_obs_LFC), axis=1))
screen_pairs_w_lfc[:1]

,org_genepair,genepair,A1_entrez,A2_entrez,A1,A2,org_A1,org_A2,rpe1_hit,hap1_hit,hap1_early_obs_LFC,hap1_late_obs_LFC,rpe1_early_obs_LFC,rpe1_late_obs_LFC,min_hap1_lfc,min_rpe1_lfc
0,ABHD12_ABHD12B,ABHD12_ABHD12B,26090,145447,ABHD12,ABHD12B,ABHD12,ABHD12B,False,False,-0.067548,-0.130963,-0.288012,-0.300921,-0.130963,-0.300921


In [98]:
# To identify SL pairs we filtered the negative GIs to those that have observed LFC < -0.9
screen_pairs_w_lfc = screen_pairs_w_lfc.assign(
    rpe1_second_hit = screen_pairs_w_lfc.apply(lambda x: x.min_rpe1_lfc < -0.9, axis=1),
    hap1_second_hit = screen_pairs_w_lfc.apply(lambda x: x.min_hap1_lfc < -0.9, axis=1)
    )
print('# of SL based on negative GI score only (rpe1):', sum(screen_pairs_w_lfc.rpe1_hit), '/', screen_pairs_w_lfc.shape[0])
print('# of SL based on LFC threshold (rpe1):', sum(screen_pairs_w_lfc.rpe1_second_hit), '/', screen_pairs_w_lfc.shape[0])
print('# of SL based on negative GI score only (hap1):', sum(screen_pairs_w_lfc.hap1_hit), '/', screen_pairs_w_lfc.shape[0])
print('# of SL based on LFC threshold (hap1):', sum(screen_pairs_w_lfc.hap1_second_hit), '/', screen_pairs_w_lfc.shape[0])
screen_pairs_w_lfc[:1]

# of SL based on negative GI score only (rpe1): 107 / 685
# of SL based on LFC threshold (rpe1): 190 / 685
# of SL based on negative GI score only (hap1): 214 / 685
# of SL based on LFC threshold (hap1): 161 / 685


,org_genepair,genepair,A1_entrez,A2_entrez,A1,A2,org_A1,org_A2,rpe1_hit,hap1_hit,hap1_early_obs_LFC,hap1_late_obs_LFC,rpe1_early_obs_LFC,rpe1_late_obs_LFC,min_hap1_lfc,min_rpe1_lfc,rpe1_second_hit,hap1_second_hit
0,ABHD12_ABHD12B,ABHD12_ABHD12B,26090,145447,ABHD12,ABHD12B,ABHD12,ABHD12B,False,False,-0.067548,-0.130963,-0.288012,-0.300921,-0.130963,-0.300921,False,False


In [99]:
# Save the overlapping hits from rpe1_hit and rpe1_second_hit 
screen_pairs_w_lfc = screen_pairs_w_lfc.assign(
    rpe1_overlapping_hit = screen_pairs_w_lfc.apply(lambda x: x.rpe1_hit == True and x.rpe1_second_hit == True, axis=1),
    hap1_overlapping_hit = screen_pairs_w_lfc.apply(lambda x: x.hap1_hit == True and x.hap1_second_hit == True, axis=1),
    )
print('# of SL based on overlap (rpe1):', sum(screen_pairs_w_lfc.rpe1_overlapping_hit), '/', screen_pairs_w_lfc.shape[0])
print('# of SL based on overlap (hap1):', sum(screen_pairs_w_lfc.hap1_overlapping_hit), '/', screen_pairs_w_lfc.shape[0])

# of SL based on overlap (rpe1): 28 / 685
# of SL based on overlap (hap1): 56 / 685


In [100]:
# Extract gene pairs that have negative GI score (rpe1_hit == True), but doesn't meet the threshold (min_rpe1_lfc > -0.9).
rpe1_removed_pairs = screen_pairs_w_lfc.loc[(screen_pairs_w_lfc.rpe1_overlapping_hit == True) & (screen_pairs_w_lfc.min_rpe1_lfc > -0.9), ]
display(rpe1_removed_pairs)

# Extract gene pairs that have negative GI score (rpe1_hit == True), but doesn't meet the threshold (min_rpe1_lfc > -0.9).
hap1_removed_pairs = screen_pairs_w_lfc.loc[(screen_pairs_w_lfc.hap1_overlapping_hit == True) & (screen_pairs_w_lfc.min_hap1_lfc > -0.9), ]
display(hap1_removed_pairs)

,org_genepair,genepair,A1_entrez,A2_entrez,A1,A2,org_A1,org_A2,rpe1_hit,hap1_hit,hap1_early_obs_LFC,hap1_late_obs_LFC,rpe1_early_obs_LFC,rpe1_late_obs_LFC,min_hap1_lfc,min_rpe1_lfc,rpe1_second_hit,hap1_second_hit,rpe1_overlapping_hit,hap1_overlapping_hit


,org_genepair,genepair,A1_entrez,A2_entrez,A1,A2,org_A1,org_A2,rpe1_hit,hap1_hit,hap1_early_obs_LFC,hap1_late_obs_LFC,rpe1_early_obs_LFC,rpe1_late_obs_LFC,min_hap1_lfc,min_rpe1_lfc,rpe1_second_hit,hap1_second_hit,rpe1_overlapping_hit,hap1_overlapping_hit


In [101]:
screen_pairs_w_lfc[:3]

,org_genepair,genepair,A1_entrez,A2_entrez,A1,A2,org_A1,org_A2,rpe1_hit,hap1_hit,hap1_early_obs_LFC,hap1_late_obs_LFC,rpe1_early_obs_LFC,rpe1_late_obs_LFC,min_hap1_lfc,min_rpe1_lfc,rpe1_second_hit,hap1_second_hit,rpe1_overlapping_hit,hap1_overlapping_hit
0,ABHD12_ABHD12B,ABHD12_ABHD12B,26090,145447,ABHD12,ABHD12B,ABHD12,ABHD12B,False,False,-0.067548,-0.130963,-0.288012,-0.300921,-0.130963,-0.300921,False,False,False,False
1,ABL1_ABL2,ABL1_ABL2,25,27,ABL1,ABL2,ABL1,ABL2,True,False,0.068177,0.047309,0.059887,0.149824,0.047309,0.059887,False,False,False,False
2,ACBD3_TMED8,ACBD3_TMED8,64746,283578,ACBD3,TMED8,ACBD3,TMED8,False,False,0.228133,0.258238,-0.099709,-0.178376,0.228133,-0.178376,False,False,False,False


In [102]:
#chymera_hits = screen_pairs_w_lfc[['genepair', 'A1', 'A2', 'A1_entrez', 'A2_entrez', 'rpe1_overlapping_hit', 'hap1_overlapping_hit', 'org_A1', 'org_A2']]
#chymera_hits = screen_pairs_w_lfc[['genepair', 'A1', 'A2', 'A1_entrez', 'A2_entrez', 'rpe1_overlapping_hit']]
#chymera_hits[:3]

In [103]:
chymera_hits = screen_pairs_w_lfc[['genepair', 'A1', 'A2', 'A1_entrez', 'A2_entrez', 'rpe1_overlapping_hit', 'hap1_overlapping_hit',
                                    'org_A1', 'org_A2']]
chymera_hits[:3]

,genepair,A1,A2,A1_entrez,A2_entrez,rpe1_overlapping_hit,hap1_overlapping_hit,org_A1,org_A2
0,ABHD12_ABHD12B,ABHD12,ABHD12B,26090,145447,False,False,ABHD12,ABHD12B
1,ABL1_ABL2,ABL1,ABL2,25,27,False,False,ABL1,ABL2
2,ACBD3_TMED8,ACBD3,TMED8,64746,283578,False,False,ACBD3,TMED8


In [104]:
# Create a DataFrame from the original data
chymera_pairs = pd.DataFrame(chymera_hits)

# Reshape the dataset
df_rpe1 = chymera_pairs.copy()
df_rpe1['SL'] = df_rpe1['rpe1_overlapping_hit']
df_rpe1['cell_line'] = 'RPE1SS77'
df_rpe1['DepMap_ID'] = 'ACH-002463'
df_rpe1 = df_rpe1[['genepair', 'A1', 'A2', 'A1_entrez', 'A2_entrez', 'DepMap_ID', 'cell_line', 'SL', 'org_A1', 'org_A2']]

df_hap1 = chymera_pairs.copy()
df_hap1['SL'] = df_hap1['hap1_overlapping_hit']
df_hap1['cell_line'] = 'HAP1'
df_hap1['DepMap_ID'] = 'ACH-002475'
df_hap1 = df_hap1[['genepair', 'A1', 'A2', 'A1_entrez', 'A2_entrez', 'DepMap_ID', 'cell_line', 'SL', 'org_A1', 'org_A2']]

# Concatenate the two DataFrames
chymera_pairs_df = pd.concat([df_rpe1, df_hap1], ignore_index=True)
chymera_pairs_df.sort_values(by=['genepair'], inplace=True)
chymera_pairs_df = chymera_pairs_df.reset_index(drop=True)

In [105]:
chymera_pairs_df.loc[chymera_pairs_df.isna().any(axis=1), :]

,genepair,A1,A2,A1_entrez,A2_entrez,DepMap_ID,cell_line,SL,org_A1,org_A2


In [106]:
chymera_pairs_df.loc[chymera_pairs_df['org_A1'] == 'FAM102A', ]

,genepair,A1,A2,A1_entrez,A2_entrez,DepMap_ID,cell_line,SL,org_A1,org_A2
350,EEIG1_EEIG2,EEIG1,EEIG2,399665,284611,ACH-002475,HAP1,False,FAM102A,FAM102B
351,EEIG1_EEIG2,EEIG1,EEIG2,399665,284611,ACH-002463,RPE1SS77,True,FAM102A,FAM102B


In [107]:
# Function to sort each pair of gene symbols and their Entrez IDs
def sort_gene_pairs(row):
    # Sort the genes alphabetically and determine new order
    sorted_genes = natsorted([row['A1'], row['A2']])
    
    # Match the sorted genes to the original ones and rearrange Entrez IDs accordingly
    if sorted_genes[0] == row['A1']:
        return pd.Series([sorted_genes[0], sorted_genes[1], row['A1_entrez'], row['A2_entrez']])
    else:
        return pd.Series([sorted_genes[0], sorted_genes[1], row['A2_entrez'], row['A1_entrez']])

# Apply the sorting to each row
df = chymera_pairs_df.copy()
df[['A1_sorted', 'A2_sorted', 'A1_entrez_sorted', 'A2_entrez_sorted']] = df.apply(sort_gene_pairs, axis=1)

# Drop the old columns and rename the new ones
chymera_pairs_df_new = df.drop(columns=['A1', 'A2', 'A1_entrez', 'A2_entrez']).copy()
chymera_pairs_df_new = chymera_pairs_df_new.rename(columns={
    'A1_sorted': 'A1',
    'A2_sorted': 'A2',
    'A1_entrez_sorted': 'A1_entrez',
    'A2_entrez_sorted': 'A2_entrez'
})

In [108]:
chymera_pairs_df_new = chymera_pairs_df_new[['genepair', 'A1', 'A2', 'A1_entrez', 'A2_entrez', 'DepMap_ID', 'cell_line', 'SL', 'org_A1', 'org_A2']]
chymera_pairs_df_new

,genepair,A1,A2,A1_entrez,A2_entrez,DepMap_ID,cell_line,SL,org_A1,org_A2
0,ABHD12_ABHD12B,ABHD12,ABHD12B,26090,145447,ACH-002463,RPE1SS77,False,ABHD12,ABHD12B
1,ABHD12_ABHD12B,ABHD12,ABHD12B,26090,145447,ACH-002475,HAP1,False,ABHD12,ABHD12B
2,ABL1_ABL2,ABL1,ABL2,25,27,ACH-002475,HAP1,False,ABL1,ABL2
3,ABL1_ABL2,ABL1,ABL2,25,27,ACH-002463,RPE1SS77,False,ABL1,ABL2
4,ACBD3_TMED8,ACBD3,TMED8,64746,283578,ACH-002463,RPE1SS77,False,ACBD3,TMED8
...,...,...,...,...,...,...,...,...,...,...
1365,ZMIZ1_ZMIZ2,ZMIZ1,ZMIZ2,57178,83637,ACH-002463,RPE1SS77,False,ZMIZ2,ZMIZ1
1366,ZNF503_ZNF703,ZNF503,ZNF703,84858,80139,ACH-002475,HAP1,False,ZNF503,ZNF703
1367,ZNF503_ZNF703,ZNF503,ZNF703,84858,80139,ACH-002463,RPE1SS77,False,ZNF503,ZNF703
1368,ZNF608_ZNF609,ZNF608,ZNF609,57507,23060,ACH-002463,RPE1SS77,False,ZNF609,ZNF608


In [109]:
chymera_pairs_df_new.to_csv(file_path_processed_chymera_df, index = False)